# Part 3: Advanced Topics in Causality

The final two chapters cover advanced causal inference, including cutting-edge research on algorithmic bias, model fairness, and reinforcement learning.

# 8: Model Fairness

Widespread ML adoption significantly affects lives: from social media feeds to credit and health decisions—making ethical development essential.

Given extensive ML ethics literature, we assume reader familiarity, define key terms, and focus on using causality to measure and mitigate algorithmic bias.

This section explains how causal methods assess and create fair models, highlighting their superiority over standard statistics in detecting discrimination.

## 8.1: Introduction to Model Fairness

### 8.1.1: Prerequisite Knowledge

This chapter assumes readers understand algorithmic bias.
For those unfamiliar, see Chapter 3 of the *fast.ai* book (Howard & Gugger, 2020).

Mehrabi et al. (2019) offer a comprehensive survey of bias types in machine learning, clearly distinguishing dataset, algorithmic, deployment, and social/environmental bias from statistical bias.
These definitions guide this chapter.

We assume prior knowledge of causal inference, specifically Chapter 3's causal estimation process.

### 8.1.2: What is Model Fairness?

Algorithmic bias is often used broadly to describe unethical tech issues, such as biased algorithms or safety systems favouring certain groups.
While this raises awareness, it lacks precision for machine learning-specific concerns, as noted by Liberties EU.

This chapter examines *model fairness*, focusing on how machine learning systems may perpetuate discrimination or show varying performance across groups.
It draws on Mehrabi et al. (2019), section 4, which defines fairness at individual (similar people should receive similar outcomes) and group (equal treatment across groups) levels.

## 8.2: How is Fairness Measured?

Mehrabi et al. (2019) and Caton and Haas (2020) categorise fairness quantification methods.
While some approaches, like *Fairness Through Unawareness* (Kusner et al., 2017), claim fairness by ignoring protected attributes, most recognise models can indirectly use these attributes via other features.
Common definitions include *Demographic Parity* and *Equality of Opportunity* (Kusner et al., 2017).

Conditional statistical parity ensures similar outcomes between protected groups after accounting for relevant factors.

Some fairness measures assess prediction similarity for similar people, while others evaluate model calibration across different groups.

Fairness measurement adoption remains limited, with inconsistent metric selection across use cases.
Few practitioners use fairness methods, and those who do employ varying techniques.
Before exploring causal fairness metrics, we explain why causal approaches are preferable.

## 8.3: Why Use Causality?

Given many fairness evaluation methods, it's reasonable to question whether causal approaches should replace standard non-causal techniques.

Omitted variable bias can distort algorithmic fairness assessments if key covariates are unaccounted for.
 Simpson's Paradox (where effect reversals occur with different conditioning variables) can lead to misleading conclusions about discrimination.
 Defining "legitimate factors" for conditional fairness metrics is debatable, and their inclusion significantly alters results.
 Causal graphs map foundational assumptions for fairness analysis, providing a debatable and refineable reference point for subject experts.

Framing fairness as causal inference reveals bias mechanisms, unlike correlation which only identifies discrepancies without explaining causes.
This clarifies how algorithms propagate bias.

The reason for using causality is the same for model fairness as for any other causal inference application.

1. Causal graphs formalise assumptions about bias perpetuation in machine learning systems.
2. Treating fairness analysis as causal inference quantifies bias mechanisms, not just identifies discrepancies.

## 8.4: Causal Fairness Measures


### 8.4.1: Background

Early fairness studies used standard causal metrics like Effect of Treatment on the Treated (ETT) (Kusner et al., 2017).
However, ETT cannot isolate treatment effects from confounding pathways.
Later work (Zhang & Bareinboim, 2018) introduced specific causal measures to isolate discriminatory mechanisms.

* CDE measures a treatment's (X) direct effect on an outcome (Y), holding other factors constant.
  In fairness contexts, X is a protected characteristic (e.g., gender, race).
* NDE differs from CDE by showing X's direct effect on Y when mediators W are set to their natural values under X intervention.
* NIE measures how outcome (Y) changes due to mediator (W) shifts when treatment (X) is held constant.

These metrics describe unfair treatment in fairness settings, but only apply when $X$ has no other parent in the causal graph.
Otherwise, other discrimination sources may remain undetected.

### 8.4.2: The Standard Fairness Model

Researchers studying potential model discrimination often lack access to a model's inner workings (e.g., proprietary data), hindering causal analysis.
Zhang & Bareinboim (2018) introduced the *Standard Fairness Model*—a causal framework for fairness questions, even with limited model transparency.

Standard Fairness Model causal graph ([Figure 8.1](#fig-sfm-generic)): $X$ (protected category), $Y$ (outcome), $W$ (mediators between $X$ and $Y$), $Z$ (confounders between $X$ and $Y$).

<center>
  <img
    src="images/sfm_generic.png"
    alt="Causal graph for the Standard Fairness Model"
    width="400"/>
  <a id="fig-sfm-generic"></a>
  <h6>Figure 8.1: Causal graph for the Standard Fairness Model</h6>
</center>

Zhang and Bareinboim introduce counterfactual measures for discrimination, similar to CDE/NDE/NIE but addressing their limitations.
They add a dedicated measure for spurious effects caused by confounders (e.g., $X \leftarrow Z \rightarrow Y$).

* **Ctf-DE**
  The counterfactual direct effect of treatment *X* on outcome *Y*.
  It measures *Y*'s change when *X* shifts from *x_0* to *x_1*, with mediators held at their natural values under *X* = *x_0*.
  A non-zero value proves direct disparity due to protected status.
* **Ctf-IE Summary**
  Ctf-IE measures the indirect effect of treatment *X* on outcome *Y* by shifting mediators *W* to their natural state at *X* = *x_1* while *X* remains fixed at *x_0*.
  It quantifies discrimination via backdoor paths.
* **CTF-SE Summary**
  CTF-SE quantifies the spurious effect from confounders between a protected characteristic $X$ and outcome $Y$, missing in standard causal frameworks (CDE/NDE/NIE).
  It measures discrimination via residual effects.
  When no discrimination exists, direct (DE) and indirect effects (IE) are zero, with all observed effect attributed to CTF-SE.


Formulas for these measures are below:

$C_\text{tf-DE}$, $C_\text{tf-IE}$, and $C_\text{tf-SE}$ represent differences in conditional probabilities:

$$\begin{aligned}
C_{\text{tf-DE}_{x_0, x_1}} (y \mid x) &= P(y_{x_1, W_{x_0}} \mid x) - P(y_{x_0} \ mid x) \\
C_{\text{tf-IE}_{x_0, x_1}} (y \mid x) &= P(y_{x_0, W_{z_1}} \mid x) - P(y_{x_0} \mid x) \\
C_{\text{tf-SE}_{x_0, x_1}} (y) &= P(y_{x_0} \mid x_1) - P(y \mid x_0)
\end{aligned}$$

Under the Standard Fairness Model's assumptions, these metrics can be directly estimated from observational data.
Identification formulas are provided.

$$\begin{aligned}
C_{\text{tf-DE}_{x_0,x_1}} (y \mid x) &= \sum_{z,w} \bigl[ P(y \mid X_1, z, w) - P(y \mid x_0, z, w) \bigr] P(w \mid x_0, z) P(z \mid x) \\
C_{\text{tf-IE}_{x_0,x_1}} (y \mid x) &= \sum_{z,w} P(y \mid x_0, z, w) \bigl[ P(W \mid x_1, z) - P(w \mid x_1, z) \bigr] P(z \mid x) \\
C_{\text{tf-SE}_{x_0,x_1}} (y) &= \sum_z P(y \mid x_0, z) \bigl[ P(z \mid x_0) - P(z \mid x_1) \bigr]
\end{aligned}$$

These formulas simplify causal inference of discriminatory effect from observational data under the Standard Fairness Model.

These measures decompose total observed disparities between protected groups into distinct components, quantifying both the extent and nature of any discrimination in the data.

$$TV_{x_0,x_1}(y) = DE_{x_0,x_1}$(y mid x_0) − SE_{x_1,x_0}(y) − IE_{x_1,x_0}(y \mid x_0)$$

### 8.4.3: A Framework for Causal Fairness Analysis

Plečko and Bareinboim extend the Standard Fairness Model with a standardised causal process for assessing model fairness.
See their paper (*Plečko and Bareinboim 2022*) and the [Fairness Cookbook website](https://fairness.causalai.net/) for details.

Plečko and Bareinboim formalise the *Fundamental Problem of Causal Fairness Analysis*: breaking down outcome variations into causal measures.
They organise existing metrics (including ETT, CDE, NDE, NIE, Ctf-DE/IE/SE) into a hierarchy by population granularity (total to individual) and disparity mechanism (causal, spurious, direct, indirect).
Their Fairness Map illustrates metric relationships, dependencies, and impossible quantifications (see Fairness Map).

<center>
  <img
    src="images/fairness-map.png"
    alt="Causal graph for the Standard Fairness Model"
    width="400"/>
</center>

#### 8.4.3.1: The Fairness Cookbook

Plečko and Bareinboim's *Fairness Cookbook* provides a step-by-step guide for analysts using causal tools to assess fairness.
Full examples are in their 2022 work.

1. Obtain the dataset
2. Determine the Standard Fairness Model projection
   * Identify dataset variables mapping to sets $W$, $Z$, treatment $X$, and outcome $Y$.
   * Check for bidirected edges between variables; if present, further work is needed to estimate causal impacts.
3. Assess disparate treatment
   * Use $\text{-DE}$ methods to quantify the direct effect.
4. Assess disparate impact
   * Use an $\text{-IE}$ method to quantify indirect effects.

## 8.5: Fairness Case Study: Identifying Bias in the COMPAS Recidivism Model

This case study uses the COMPAS dataset, an infamous model exposed by ProPublica for racial bias against Black defendants, leading to longer sentences compared to white defendants.

### 8.5.1: Framing the Problem + Non-Causal Estimates

ProPublica's analysis found the algorithm predicted higher recidivism risk for Black defendants than White defendants, with [Figure 8.2](#fig-compas-pred) showing the disparity in 'medium' or 'high' risk classifications.

<center>
  <img
    src="images/compas_predictions_by_race.png"
    alt="Difference in COMPAS Recidivism Predictions by Race"
    width="400"/>
  <a id="fig-compas-pred"></a>
  <h6>Figure 8.2: Difference in COMPAS Recidivism Predictions by Race</h6>
</center>

Black defendants are 56.6% more likely than similar white defendants to be classified as medium or high risk, after controlling for sex, age, charge type (felony/misdemeanour), and prior convictions. 
This disparity in model output, used directly in sentencing, is concerning.

Model fairness can be assessed by examining performance metrics for disparities across groups. 
[Figure 8.3](#fig-compas-acc) shows significant racial differences in outcomes.

<center>
  <img
    src="images/compas_accuracy_by_race.png"
    alt="Difference in COMPAS Model Accuracy by Race"
    width="400"/>
  <a id="fig-compas-acc"></a>
  <h6>Figure 8.3: Difference in COMPAS Model Accuracy by Race</h6>
</center>

A non-causal analysis shows the model predicts risk accurately 10.5% less often for Black defendants than White defendants, indicating poorer performance with Black defendant data.

### 8.5.2: Causal Measures

The previous section shows two approaches to framing fairness in models, focusing on one or both of these:

1. Model predictions
2. Performance metrics: accuracy, precision, recall

Linear models cannot confirm causal relationships. 
Causal inference requires a causal graph, such as the COMPAS graph (Plečko & Bareinboim 2022, [Figure 8.4](#fig-compas-graph)).

<center>
  <img
    src="images/compas_graph.png"
    alt="Causal Graph for COMPAS Predictions"
    width="400"/>
  <a id="fig-compas-graph"></a>
  <h6>Figure 8.4: Causal Graph for COMPAS Predictions</h6>
</center>

After graph construction, the key edge between `race` and `predicted_recid` measures direct discriminatory impact. 
Causal estimates (using `DoWhy`) differ from non-causal ones, as shown in the comparison table.

 mid  Outcome                mid  Causal    mid  Non-causal  mid 
 mid ----------------------- mid ---------- mid ------------ mid 
 mid  Predicted recidivism   mid  +21.8%    mid  +56.6%      mid 
 mid  Model accuracy         mid  -11.6%    mid  -10.5%      mid 

Both approaches confirm COMPAS has quantifiable racial bias, though they differ on race's specific contribution. 
Unlike Simpson's Paradox, trends don't reverse, and causal analysis boosts confidence in measuring true effects.

## 8.6: Additional Topics

### 8.6.1: Enforcing Fairness During Model Training

Recent work integrates fairness directly into model building, rather than just assessing unfairness.
Di Stefano et al. (2020) introduce a loss penalty based on the controlled direct effect (CDE) to minimise disparity.
As shown in [Figure 8.5](#fig-mfcde-dag), their causal model uses $Z$ (protected attribute), $Y$ (task label), and **$X$** (covariates).

<center>
  <img
    src="images/mfcde.png"
    alt="Causal graph for MFCDE"
    width="400"/>
  <a id="fig-mfcde-dag"></a>
  <h6>Figure 8.5: Causal graph for MFCDE</h6>
</center>

A key assumption is no confounding variables between protected attributes (e.g., race) and outcomes (e.g., loan defaults). 
This often holds in fairness contexts but must be verified per application.

The authors frame fair model training as avoiding learning the conditional distributional equality (CDE) between protected attribute $Z$ and outcome $Y$. 
To address differentiability requirements, they use a surrogate model's propensity scores with mean-field CDE (MFCDE) minimisation alongside the task loss (e.g., binary cross-entropy). 
This combines the original loss $\mathcal{L}_o$, a differentiable fairness penalty $\mathcal{R}_f$, and a regularisation weight $\lambda$:
$\mathcal{L} = \mathcal{L}_o + \lambda \mathcal{R}_f$.

$$ \mathcal{L}_f = (1-\lambda)\mathcal{L}_o + \lambda\mathcal{R}_f $$

The fairness penalty uses propensity matching to iteratively calculate MFCDE during training. 
A surrogate model predicts the output using propensity scores and the protected attribute, enabling gradient-based optimisation of the fair loss.

This approach requires updating the surrogate model at each iteration, increasing computational cost through both training and new training data. 
However, minimising MFCDE reduces the CDE measuring disparity. 
For higher $\lambda$, pre-training with smaller $\lambda$ values before gradually increasing to the target level is recommended.

### 8.6.2: Fairness in Reinforcement Learning

Fairness assessment for trained RL models can use the Fairness Cookbook (Section 8.4.3).
However, RL's distinct training dynamics require further fairness coverage in Chapter 9.
